In [279]:
#cleaning

import pandas as pd
import numpy as np

def process_factor_returns(input_file: str, output_file: str) -> pd.DataFrame:
    """
    Reads daily factor returns CSV, sorts chronologically, aggregates daily returns
    to monthly geometrically compounded returns (keyed by the last day of each month), 
    exports to a new CSV, and runs dynamic sanity checks.
    """
    print(f"Loading data from '{input_file}'...")
    
    # 1. Load raw CSV file
    # 'skiprows=1' skips the dataset title header 'factor_returns' on line 1
    df = pd.read_csv(input_file, skiprows=1)
    
    # 2. Parse dates & sort chronologically
    df['datetime'] = pd.to_datetime(df['datetime'], format='%d-%m-%Y')
    df = df.sort_values('datetime').reset_index(drop=True)
    
    # 3. Ensure numeric data types for all factor columns
    factor_cols = [col for col in df.columns if col != 'datetime']
    df[factor_cols] = df[factor_cols].apply(pd.to_numeric, errors='coerce')
    
    # 4. Resample to Monthly frequency by geometrically compounding daily returns
    def compound(series):
        return np.prod(1 + series) - 1

    try:
        monthly_df = df.set_index('datetime').resample('ME').apply(compound).reset_index()
    except ValueError:
        # Fallback for older pandas versions
        monthly_df = df.set_index('datetime').resample('M').apply(compound).reset_index()
        
    # Format 'datetime' column as YYYY-MM-DD (Last calendar day of each month)
    monthly_df['datetime'] = monthly_df['datetime'].dt.strftime('%Y-%m-%d')
    
    # 5. Export to new CSV file
    monthly_df.to_csv(output_file, index=False)
    print(f"Successfully saved monthly aggregated factor returns to '{output_file}'.\n")
    
    # ==========================================
    # Sanity Checks
    # ==========================================
    print("=" * 55)
    print("                SANITY CHECKS                ")
    print("=" * 55)
    
    # Check 1: Missing values
    missing_count = monthly_df.isna().sum().sum()
    print(f"[Check 1/5] Missing/NaN values: {missing_count} | Passed: {missing_count == 0}")
    
    # Check 2: Chronological order
    is_ordered = monthly_df['datetime'].is_monotonic_increasing
    print(f"[Check 2/5] Chronological monotonicity: Passed = {is_ordered}")
    
    # Check 3: Date bounds & total periods
    start_date = monthly_df['datetime'].iloc[0]
    end_date = monthly_df['datetime'].iloc[-1]
    total_months = len(monthly_df)
    print(f"[Check 3/5] Date Range: {start_date} -> {end_date} | Total Months: {total_months}")
    
    # Check 4: Dynamic Spot check - First available month & first factor
    first_month_end = monthly_df['datetime'].iloc[0]
    first_factor = factor_cols[0]
    first_month_start = pd.to_datetime(first_month_end).replace(day=1).strftime('%Y-%m-%d')
    
    daily_first = df[(df['datetime'] >= first_month_start) & (df['datetime'] <= first_month_end)][first_factor]
    compound_first = np.prod(1 + daily_first) - 1
    monthly_first = monthly_df.loc[monthly_df['datetime'] == first_month_end, first_factor].values[0]
    diff_first = abs(compound_first - monthly_first)
    print(f"[Check 4/5] Spot Check ({first_month_end[:7]} '{first_factor}' compound match): Diff = {diff_first:.12f}")
    
    # Check 5: Dynamic Spot check - Final available month & last factor
    last_month_end = monthly_df['datetime'].iloc[-1]
    last_factor = factor_cols[-1]
    last_month_start = pd.to_datetime(last_month_end).replace(day=1).strftime('%Y-%m-%d')
    
    daily_last = df[(df['datetime'] >= last_month_start) & (df['datetime'] <= last_month_end)][last_factor]
    compound_last = np.prod(1 + daily_last) - 1
    monthly_last = monthly_df.loc[monthly_df['datetime'] == last_month_end, last_factor].values[0]
    diff_last = abs(compound_last - monthly_last)
    print(f"[Check 5/5] Spot Check ({last_month_end[:7]} '{last_factor}' compound match): Diff = {diff_last:.12f}")
    print("=" * 55)
    
    return monthly_df

if __name__ == '__main__':
    # Update the input_file string here to run on your new file
    monthly_returns_df = process_factor_returns(
        input_file='factor_returns_daily.csv', 
        output_file='monthly_factor_returns.csv'
    )

Loading data from 'factor_returns_daily.csv'...
Successfully saved monthly aggregated factor returns to 'monthly_factor_returns.csv'.

                SANITY CHECKS                
[Check 1/5] Missing/NaN values: 0 | Passed: True
[Check 2/5] Chronological monotonicity: Passed = True
[Check 3/5] Date Range: 2008-01-31 -> 2026-07-31 | Total Months: 223
[Check 4/5] Spot Check (2008-01 'BFSICOMP_V2_INTM' compound match): Diff = 0.000000000000
[Check 5/5] Spot Check (2026-07 'value' compound match): Diff = 0.000000000000


In [280]:
# final data before backtest

"""
Merge regime classification columns (Regime_Output.xlsx) with theme/factor
monthly returns (monthly_factor_returns.csv) into a single aligned CSV,
restricted to May 2014 - May 2026.

Structure, in order, and why:
  1. Load + minimal cleaning (dates, octant label format)
  2. Sort chronologically
  3. Restrict to the target period
  4. Sanity checks on that restricted period
  5. Merge
  6. Lag + octant-validity mask
  7. Calculations/transformations (across-relative, self-relative)
  8. Verification
  9. Save
Sorting and picking the period happen BEFORE any calculation or
transformation -- this is what caught the bug below in the first place.
Doing a transformation (theme_mean) before the period was chosen is exactly
how a stray 2008-2014 tail ended up inside a number that was supposed to
describe 2015-2026 only.

FIX: theme_mean for the *_theme_relative ("Self Relative") columns must be
computed ONLY over the months that actually feed one of the 6 valid
Backtest_Octant buckets -- not over the full monthly_factor_returns.csv
history (2008-2026), and not even over the full 2015-2026 classifier window,
since 5 of those 137 months belong to the 2 excluded low-N octants. The old
code computed factors[col].mean() before merging with regime data at all
and before the date restriction, so it silently used the full unrestricted
2008-2026 sample as the baseline -- a different, larger population than the
one being decomposed into 6 octant buckets. That's why Sentiment (Self
Relative) came out negative in 5 of 6 octants: there's no way for a baseline
computed on the SAME set of months being decomposed to land on one side of
every subgroup, but a baseline computed on a different, larger set can.

Also note the 1-month lag: Backtest_Octant observed at month t governs the
return realized at month t+1 everywhere else in this pipeline (see
get_1m_returns / run_regime_factor_analysis). So "months in the 6 octants
we're using" is defined here on the LAGGED octant label, matching the exact
set of return-months every octant dashboard block draws from.
"""

import pandas as pd
import numpy as np

START = "2014-05-31"
END = "2026-05-31"

REGIME_FILE = "../Macro Scores/V7_Regime_Output.xlsx"
FACTOR_FILE = "monthly_factor_returns.csv"
OUTPUT_FILE = "regime_factor_merged.csv"

REGIME_COLS = ["Base_Regime_Confirmed", "Amplifier_Direction", "Backtest_Octant"]
FACTOR_COLS = [
    "earnings", "management", "momentum", "profitability",
    "quality", "sentiment main", "Reversal", "value",
]

# The 6 octants with adequate N -- Deflation Headwind (N=2) and
# Stagflation Headwind (N=3) excluded, same call made everywhere else in
# this project.
VALID_OCTANTS = [
    "Goldilocks Tailwind",
    "Goldilocks Headwind",
    "Reflation Headwind",
    "Reflation Tailwind",
    "Stagflation Tailwind",
    "Deflation Tailwind",
]

# ===========================================================================
# 1. Load + minimal cleaning
# ===========================================================================
regime = pd.read_excel(REGIME_FILE, sheet_name="Sheet1")
regime["Date"] = pd.to_datetime(regime["Date"]).dt.normalize() + pd.offsets.MonthEnd(0)
regime = regime[["Date"] + REGIME_COLS]

# Backtest_Octant in the source file is "Goldilocks + Tailwind"; normalize
# to match VALID_OCTANTS / REGIME_ORDER's " " separator used throughout the
# rest of this notebook's sheets and labels.
regime["Backtest_Octant"] = regime["Backtest_Octant"].str.replace(" + ", " ", regex=False)

factors = pd.read_csv(FACTOR_FILE)
factors["datetime"] = pd.to_datetime(factors["datetime"]).dt.normalize() + pd.offsets.MonthEnd(0)
factors = factors.rename(columns={"datetime": "Date"})

# ===========================================================================
# 2. Sort chronologically -- before any restriction or calculation
# ===========================================================================
regime = regime.sort_values("Date").reset_index(drop=True)
factors = factors.sort_values("Date").reset_index(drop=True)

# ===========================================================================
# 3. Restrict to the target period -- before any calculation
# ===========================================================================
mask_r = (regime["Date"] >= START) & (regime["Date"] <= END)
mask_f = (factors["Date"] >= START) & (factors["Date"] <= END)
regime = regime.loc[mask_r].reset_index(drop=True)
factors = factors.loc[mask_f].reset_index(drop=True)

# ===========================================================================
# 4. Sanity checks on the restricted period
# ===========================================================================
full_index = pd.date_range(START, END, freq="ME")
missing_in_regime = set(full_index) - set(regime["Date"])
missing_in_factors = set(full_index) - set(factors["Date"])
if missing_in_regime:
    print(f"WARNING: {len(missing_in_regime)} month(s) missing from Regime_Output.xlsx: "
          f"{sorted(d.date() for d in missing_in_regime)}")
if missing_in_factors:
    print(f"WARNING: {len(missing_in_factors)} month(s) missing from monthly_factor_returns.csv: "
          f"{sorted(d.date() for d in missing_in_factors)}")

dup_r = regime["Date"].duplicated().sum()
dup_f = factors["Date"].duplicated().sum()
if dup_r:
    print(f"WARNING: {dup_r} duplicate date(s) in Regime_Output.xlsx")
if dup_f:
    print(f"WARNING: {dup_f} duplicate date(s) in monthly_factor_returns.csv")

# ===========================================================================
# 5. Merge (both inputs already sorted + restricted to the target period)
# ===========================================================================
merged = pd.merge(regime, factors, on="Date", how="inner").sort_values("Date").reset_index(drop=True)

# ===========================================================================
# 6. Lag + octant-validity mask
# ===========================================================================
merged["Lagged_Octant"] = merged["Backtest_Octant"].shift(1)
valid_mask = merged["Lagged_Octant"].isin(VALID_OCTANTS)
print(f"\nMonths used for the Self-Relative baseline (lagged octant in the "
      f"6 valid buckets): {valid_mask.sum()} / {len(merged)}")

# ===========================================================================
# 7. Calculations / transformations
# ===========================================================================
ACROSS_RELATIVE_BASE_COLS = [col for col in FACTOR_COLS if col != "Reversal" and col != "sentiment main"]
SELF_RELATIVE_BASE_COLS = [col for col in FACTOR_COLS]

# Cross-sectional mean -- ROW-WISE (axis=1) across themes in the SAME month.
# Not a historical/time-series baseline, so it isn't affected by the bug
# above and needs no period-gating.
merged["factor_mean_across"] = merged[ACROSS_RELATIVE_BASE_COLS].mean(axis=1)

ACROSS_RELATIVE_COLS = []
SELF_RELATIVE_COLS = []

for col in ACROSS_RELATIVE_BASE_COLS:
    rel_col_name = f"{col}_across_relative"
    merged[rel_col_name] = merged[col] - merged["factor_mean_across"]
    ACROSS_RELATIVE_COLS.append(rel_col_name)

theme_means = {}
for col in SELF_RELATIVE_BASE_COLS:
    rel_col_name = f"{col}_theme_relative"

    # Scalar historical mean for this theme, restricted to the 6-octant
    # universe (was: factors[col].mean() over the full, unmerged,
    # unrestricted 2008-2026 file -- the bug).
    theme_mean = merged.loc[valid_mask, col].mean()
    theme_means[col] = theme_mean

    merged[rel_col_name] = merged[col] - theme_mean
    SELF_RELATIVE_COLS.append(rel_col_name)

print("\nSelf-relative baselines (theme_mean), computed on the 6-octant universe only:")
for col, m in theme_means.items():
    print(f"  {col:16s}: {m:+.6f}")

# ===========================================================================
# 8. Verification
# ===========================================================================
# N-weighted average of each theme's self-relative return, across just the
# 6 valid octants, should now be ~0 by construction -- not exactly 0 to
# machine precision if a theme has scattered NaNs shifting denominators
# slightly, but nowhere near the systematic all-one-side result the old
# baseline produced.
print("\nVerification -- N-weighted avg Self-Relative return across the 6 octants "
      "(should be ~0 for every theme):")
for col in SELF_RELATIVE_BASE_COLS:
    rel_col_name = f"{col}_theme_relative"
    sub = merged.loc[valid_mask, ["Lagged_Octant", rel_col_name]].dropna()
    weighted_avg = sub[rel_col_name].mean()  # equal-weighted across valid-octant months = N-weighted across octants
    print(f"  {col:16s}: {weighted_avg:+.8f}")

# ===========================================================================
# 9. Save
# ===========================================================================
merged = merged.drop(columns=["Lagged_Octant", "factor_mean_across"])
merged = merged.rename(columns={"Date": "Month_End"})
merged = merged[["Month_End"] + REGIME_COLS + FACTOR_COLS + ACROSS_RELATIVE_COLS + SELF_RELATIVE_COLS]

merged.to_csv(OUTPUT_FILE, index=False)

print(f"\nMerged shape: {merged.shape}")
print(f"Date range in output: {merged['Month_End'].min().date()} to {merged['Month_End'].max().date()}")
print(f"Saved to: {OUTPUT_FILE}")


Months used for the Self-Relative baseline (lagged octant in the 6 valid buckets): 131 / 145

Self-relative baselines (theme_mean), computed on the 6-octant universe only:
  earnings        : +0.002432
  management      : +0.002410
  momentum        : +0.001280
  profitability   : -0.000121
  quality         : +0.000467
  sentiment main  : +0.006703
  Reversal        : +0.010578
  value           : +0.002132

Verification -- N-weighted avg Self-Relative return across the 6 octants (should be ~0 for every theme):
  earnings        : +0.00000000
  management      : -0.00000000
  momentum        : -0.00000000
  profitability   : -0.00000000
  quality         : +0.00000000
  sentiment main  : -0.00000000
  Reversal        : +0.00000000
  value           : -0.00000000

Merged shape: (145, 26)
Date range in output: 2014-05-31 to 2026-05-31
Saved to: regime_factor_merged.csv


In [281]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr

def calculate_max_drawdown(returns: pd.Series) -> float:
    """Calculates the maximum drawdown of a cumulative return series."""
    if returns.empty or returns.isna().all():
        return np.nan
    cum_ret = (1 + returns).cumprod()
    peak = cum_ret.cummax()
    drawdown = (cum_ret - peak) / peak
    return drawdown.min()

def fwd_compound(series: pd.Series, window: int) -> pd.Series:
    """Calculates geometrically compounded forward return over N periods."""
    result = pd.Series(index=series.index, dtype=float)
    for i in range(len(series)):
        if i + window <= len(series):
            result.iloc[i] = np.prod(1 + series.iloc[i : i + window]) - 1
        else:
            result.iloc[i] = np.nan
    return result

def run_regime_factor_analysis(
    filepath: str = "regime_factor_merged.csv",
    date_col: str = "Month_End",
    regime_col: str = "Base_Regime_Confirmed",
    target_regime: str = "Goldilocks",
    factor_col: str = "momentum",
    score_col: str = None,
    weight_col: str = None
) -> pd.DataFrame:
    """
    Loads regime and factor data, applies a 1-month lag to regime signals,
    calculates 1M, 3M, and 6M geometrically compounded forward returns,
    and returns key performance statistics.
    """
    # 1. Load Data
    df = pd.read_csv(filepath)
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col).reset_index(drop=True)

    # Sanity Checks
    assert df[date_col].is_monotonic_increasing, "Dates must be strictly chronological."
    assert regime_col in df.columns, f"Regime column '{regime_col}' not found in CSV."
    assert factor_col in df.columns, f"Factor column '{factor_col}' not found in CSV."

    # 2. Apply 1-Month Lag on Regime (e.g., Jan 31 regime applies to Feb returns)
    df['Lagged_Regime'] = df[regime_col].shift(1)

    # 3. Compute Multi-Horizon Compounded Forward Returns across full timeline
    df['Fwd_1M'] = df[factor_col]
    df['Fwd_3M'] = fwd_compound(df[factor_col], 3)
    df['Fwd_6M'] = fwd_compound(df[factor_col], 6)

    # Optional Turnover calculation if portfolio weights column is present
    if weight_col and weight_col in df.columns:
        df['Turnover'] = df[weight_col].diff().abs()
    else:
        df['Turnover'] = np.nan

    # 4. Filter by Target Lagged Regime
    sub_df = df[df['Lagged_Regime'] == target_regime].copy()

    if sub_df.empty:
        available = df['Lagged_Regime'].dropna().unique().tolist()
        raise ValueError(f"No periods matched regime '{target_regime}'. Available regimes: {available}")

    # 5. Compute Statistics for 1M, 3M, and 6M Horizons
    horizons = [('1M', 'Fwd_1M', 12), ('3M', 'Fwd_3M', 4), ('6M', 'Fwd_6M', 2)]
    output_rows = []

    for h_label, col_name, periods_per_yr in horizons:
        valid = sub_df.dropna(subset=[col_name])
        rets = valid[col_name]
        
        count = len(rets)
        avg_ret = rets.mean() if count > 0 else np.nan
        ann_ret = ((1 + avg_ret) ** periods_per_yr) - 1 if (not np.isnan(avg_ret) and avg_ret > -1) else np.nan
        hit_rate = (rets > 0).mean() if count > 0 else np.nan
        ann_vol = (rets.std() * np.sqrt(periods_per_yr)) if count > 1 else np.nan
        ret_vol_ratio = (avg_ret * np.sqrt(periods_per_yr) / rets.std()) if (count > 1 and rets.std() > 0) else np.nan
        max_dd = calculate_max_drawdown(rets)

        # Optional Information Coefficient (Rank Correlation) if factor z-score is supplied
        ic = np.nan
        if score_col and score_col in valid.columns:
            scores = valid[score_col].shift(1).dropna()
            aligned_rets = valid.loc[scores.index, col_name]
            if len(scores) > 2:
                ic, _ = spearmanr(scores, aligned_rets)

        avg_turnover = valid['Turnover'].mean() if not valid['Turnover'].isna().all() else np.nan

        output_rows.append({
            'Horizon': h_label,
            'Sample Size (N)': count,
            'Avg Period Return': f"{avg_ret * 100:+.2f}%" if not np.isnan(avg_ret) else "N/A",
            'Annualized Return': f"{ann_ret * 100:+.2f}%" if not np.isnan(ann_ret) else "N/A",
            'Hit Rate': f"{hit_rate * 100:.1f}%" if not np.isnan(hit_rate) else "N/A",
            'Annualized Volatility': f"{ann_vol * 100:.2f}%" if not np.isnan(ann_vol) else "N/A",
            'Return / Vol': f"{ret_vol_ratio:.2f}" if not np.isnan(ret_vol_ratio) else "N/A",
            'Max Drawdown': f"{max_dd * 100:.2f}%" if not np.isnan(max_dd) else "N/A",
            'Information Coefficient': f"{ic:.3f}" if not np.isnan(ic) else "N/A",
            'Avg Turnover': f"{avg_turnover * 100:.2f}%" if not np.isnan(avg_turnover) else "N/A"
        })

    return pd.DataFrame(output_rows)

# ==============================================================================
# EXECUTION & CONFIGURATION
# ==============================================================================
if __name__ == "__main__":
    
    # Configuration - change these values as needed:
    FILE_PATH = "regime_factor_merged.csv"
    REGIME_COLUMN = "Base_Regime_Confirmed"  # Options: 'Base_Regime_Confirmed', 'Backtest_Octant', 'Amplifier_Direction'
    TARGET_REGIME = "Goldilocks"            # Options: 'Goldilocks', 'Reflation', 'Deflation', 'Stagflation'
    FACTOR_COLUMN = "value"               # Options: 'momentum', 'value', 'quality', 'earnings', 'profitability', 'Reversal', 'management', 'sentiment main'

    results_table = run_regime_factor_analysis(
        filepath=FILE_PATH,
        date_col="Month_End",
        regime_col=REGIME_COLUMN,
        target_regime=TARGET_REGIME,
        factor_col=FACTOR_COLUMN
    )

    print(f"\n==========================================================================")
    print(f" Performance Statistics: Factor '{FACTOR_COLUMN}' in Regime '{TARGET_REGIME}'")
    print(f"==========================================================================\n")
    print(results_table.to_markdown(index=False))


 Performance Statistics: Factor 'value' in Regime 'Goldilocks'

| Horizon   |   Sample Size (N) | Avg Period Return   | Annualized Return   | Hit Rate   | Annualized Volatility   |   Return / Vol | Max Drawdown   | Information Coefficient   | Avg Turnover   |
|:----------|------------------:|:--------------------|:--------------------|:-----------|:------------------------|---------------:|:---------------|:--------------------------|:---------------|
| 1M        |                36 | +0.41%              | +5.08%              | 75.0%      | 2.05%                   |           2.42 | -1.08%         | N/A                       | N/A            |
| 3M        |                36 | +1.10%              | +4.49%              | 88.9%      | 2.18%                   |           2.03 | -2.48%         | N/A                       | N/A            |
| 6M        |                36 | +1.60%              | +3.23%              | 77.8%      | 3.30%                   |           0.97 | -17.86%        | 

In [282]:
#automation

"""
build_regime_factor_workbook_v2.py

Extends build_regime_factor_workbook.py (backtesting.ipynb, cell 3) with:

  1. Two new breakdown blocks added to EVERY existing theme sheet, below the
     existing 6 Backtest_Octant blocks:
       - Base Regime only   (Goldilocks, Reflation, Stagflation, Deflation)
       - Amplifier Direction only (Tailwind, Headwind)
     Same 1M/3M/6M stats engine, just filtered on a different regime column.

  2. 8 NEW sheets -- one per theme, suffixed "(Relative)" -- running the exact
     same three breakdowns (Octant / Base Regime / Amplifier Direction) on
     the *_relative return columns instead of the absolute return columns.

Sheet count: 8 absolute-return sheets + 8 relative-return sheets = 16 total.

Input:
    regime_factor_merged.csv   (Month_End, Base_Regime_Confirmed,
                                 Amplifier_Direction, Backtest_Octant, the 8
                                 absolute factor return columns, and the 8
                                 *_relative columns)

Output:
    QI-CAP_Theme_Regime_Stats.xlsx  -- 16 sheets

Usage:
    python build_regime_factor_workbook_v2.py \
        --input regime_factor_merged.csv \
        --output QI-CAP_Theme_Regime_Stats.xlsx
"""

import argparse
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import os
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, Alignment
from openpyxl.utils import get_column_letter

# ---------------------------------------------------------------------------
# 1. Regime buckets actually used at the OCTANT level (Deflation Headwind
#    N=10 and Stagflation Headwind N=2 excluded -- same call your PDF notes
#    made: "N size way too small for any meaningful testing")
# ---------------------------------------------------------------------------
REGIME_ORDER = [
    "Goldilocks Tailwind",
    "Goldilocks Headwind",
    "Reflation Headwind",
    "Reflation Tailwind",
    "Stagflation Tailwind",
    "Deflation Tailwind",
]

# Base Regime and Amplifier Direction breakdowns -- all groups have adequate
# sample size (smallest is Stagflation, N=19), so none are excluded here.
BASE_REGIMES = ["Goldilocks", "Reflation", "Stagflation", "Deflation"]
AMPLIFIER_DIRECTIONS = ["Tailwind", "Headwind"]

# ---------------------------------------------------------------------------
# 2. Theme -> CSV factor return column, absolute and relative versions
#    (Industry and Linkages excluded per your instructions; sheet order
#    follows the PDF's theme order)
# ---------------------------------------------------------------------------
THEME_TO_FACTOR_COL = {
    "Value": "value",
    "Momentum": "momentum",
    "Profitability": "profitability",
    "Quality": "quality",
    "Earnings": "earnings",
    "Sentiment": "sentiment main",
    "Reversal": "Reversal",
    "Management": "management",
}

THEME_TO_ACROSS_RELATIVE_FACTOR_COL = {
    "Value": "value_across_relative",
    "Momentum": "momentum_across_relative",
    "Profitability": "profitability_across_relative",
    "Quality": "quality_across_relative",
    "Earnings": "earnings_across_relative",
    "Management": "management_across_relative",
}

THEME_TO_SELF_RELATIVE_COL = {"Value": "value_theme_relative",
    "Momentum": "momentum_theme_relative",
    "Profitability": "profitability_theme_relative",
    "Quality": "quality_theme_relative",
    "Earnings": "earnings_theme_relative",
    "Sentiment": "sentiment main_theme_relative",
    "Reversal": "Reversal_theme_relative",
    "Management": "management_theme_relative",
}

# ---------------------------------------------------------------------------
# 3. Expectation / Conviction, transcribed from the qualitative
#    Theme/Performance/Conviction/Rationale tables in
#    QI-CAP_Theme_Hypotheses_.pdf (NOT the later percentage-breakdown
#    tables -- those use different numbers). Double check this block against
#    the PDF if anything looks off; this was typed by hand.
#
#    These hypotheses are only defined at the OCTANT level (e.g. "Goldilocks
#    Tailwind") -- there's no separate hypothesis for "Goldilocks" alone or
#    "Tailwind" alone in the source PDF, so the new Base Regime / Amplifier
#    Direction blocks below leave Expectation/Conviction blank rather than
#    inventing an aggregated view. Same dict is reused for the relative-return
#    sheets since the hypotheses describe over/under-performance, which maps
#    naturally to relative return sign.
# ---------------------------------------------------------------------------
EXPECTATION_CONVICTION = {
    "Value": {
        "Goldilocks Tailwind": ("Underperform", "Medium"),
        "Goldilocks Headwind": ("Overperform", "Medium"),
        "Reflation Headwind": ("Overperform", "Medium"),
        "Reflation Tailwind": ("Overperform", "Medium"),
        "Stagflation Tailwind": ("Overperform", "Medium"),
        "Deflation Tailwind": ("Underperform", "Medium"),
    },
    "Momentum": {
        "Goldilocks Tailwind": ("Overperform", "High"),
        "Goldilocks Headwind": ("Overperform", "High"),
        "Reflation Headwind": ("Underperform", "Low"),
        "Reflation Tailwind": ("Overperform", "High"),
        "Stagflation Tailwind": ("Underperform", "Low"),
        "Deflation Tailwind": ("Overperform", "High"),
    },
    "Profitability": {
        "Goldilocks Tailwind": ("Underperform", "Medium"),
        "Goldilocks Headwind": ("Overperform", "Medium"),
        "Reflation Headwind": ("Overperform", "High"),
        "Reflation Tailwind": ("Overperform", "Low"),
        "Stagflation Tailwind": ("Overperform", "High"),
        "Deflation Tailwind": ("Overperform", "Medium"),
    },
    "Quality": {
        "Goldilocks Tailwind": ("Underperform", "Medium"),
        "Goldilocks Headwind": ("Overperform", "Medium"),
        "Reflation Headwind": ("Overperform", "Medium"),
        "Reflation Tailwind": ("Underperform", "Low"),
        "Stagflation Tailwind": ("Overperform", "Medium"),
        "Deflation Tailwind": ("Overperform", "High"),
    },
    "Earnings": {
        "Goldilocks Tailwind": ("Overperform", "High"),
        "Goldilocks Headwind": ("Overperform", "Medium"),
        "Reflation Headwind": ("Overperform", "Medium"),
        "Reflation Tailwind": ("Overperform", "High"),
        "Stagflation Tailwind": ("Overperform", "High"),
        "Deflation Tailwind": ("Overperform", "Medium"),
    },
    "Sentiment": {
        "Goldilocks Tailwind": ("Overperform", "Medium"),
        "Goldilocks Headwind": ("Overperform", "Low"),
        "Reflation Headwind": ("Underperform", "Medium"),
        "Reflation Tailwind": ("Overperform", "High"),
        "Stagflation Tailwind": ("Underperform", "Low"),
        "Deflation Tailwind": ("Overperform", "Medium"),
    },
    "Reversal": {
        "Goldilocks Tailwind": ("Neutral", "Medium"),
        "Goldilocks Headwind": ("Neutral", "Medium"),
        "Reflation Headwind": ("Overperform", "Low"),
        "Reflation Tailwind": ("Underperform", "Medium"),
        "Stagflation Tailwind": ("Overperform", "Medium"),
        "Deflation Tailwind": ("Underperform", "Low"),
    },
    "Management": {
        "Goldilocks Tailwind": ("Underperform", "Medium"),
        "Goldilocks Headwind": ("Underperform", "Medium"),
        "Reflation Headwind": ("Overperform", "Medium"),
        "Reflation Tailwind": ("Overperform", "Low"),
        "Stagflation Tailwind": ("Overperform", "Medium"),
        "Deflation Tailwind": ("Overperform", "Medium"),
    },
}

COLUMN_HEADERS = [
    "Expectation", "Conviction", "Notes", "Sample Size",
    "Average Period Returns", "Annualized Return", "Hit Rate",
    "Annualized Volatility", "Max Drawdown", "Information Coefficient",
    "Average Turnover",
]

HORIZONS = [("1M", 1, 12), ("3M", 3, 4), ("6M", 6, 2)]  # label, window, periods/yr


# ---------------------------------------------------------------------------
# Statistics engine -- unchanged from backtesting.ipynb cell 2/3 (1-month
# regime lag, geometrically compounded forward returns, same formulas for
# annualized return / vol / hit rate / max DD / IC / turnover). regime_col
# and target_regime are parameters, so the exact same function drives the
# Octant, Base Regime, and Amplifier Direction breakdowns alike.
# ---------------------------------------------------------------------------
def calculate_max_drawdown(returns: pd.Series) -> float:
    if returns.empty or returns.isna().all():
        return np.nan
    cum_ret = (1 + returns).cumprod()
    peak = cum_ret.cummax()
    drawdown = (cum_ret - peak) / peak
    return drawdown.min()


def fwd_compound(series: pd.Series, window: int) -> pd.Series:
    result = pd.Series(index=series.index, dtype=float)
    for i in range(len(series)):
        if i + window <= len(series):
            result.iloc[i] = np.prod(1 + series.iloc[i: i + window]) - 1
        else:
            result.iloc[i] = np.nan
    return result


def run_regime_factor_analysis(
    df: pd.DataFrame,
    date_col: str = "Month_End",
    regime_col: str = "Backtest_Octant",
    target_regime: str = "Goldilocks Tailwind",
    factor_col: str = "momentum",
    score_col: str = None,
    weight_col: str = None,
) -> list[dict]:
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col).reset_index(drop=True)

    assert df[date_col].is_monotonic_increasing, "Dates must be strictly chronological."
    assert regime_col in df.columns, f"Regime column '{regime_col}' not found."
    assert factor_col in df.columns, f"Factor column '{factor_col}' not found."

    # 1-month lag: a regime read as of month t applies to returns in t+1
    df["Lagged_Regime"] = df[regime_col].shift(1)

    df["Fwd_1M"] = df[factor_col]
    df["Fwd_3M"] = fwd_compound(df[factor_col], 3)
    df["Fwd_6M"] = fwd_compound(df[factor_col], 6)

    if weight_col and weight_col in df.columns:
        df["Turnover"] = df[weight_col].diff().abs()
    else:
        df["Turnover"] = np.nan

    sub_df = df[df["Lagged_Regime"] == target_regime].copy()
    if sub_df.empty:
        available = df["Lagged_Regime"].dropna().unique().tolist()
        raise ValueError(f"No periods matched regime '{target_regime}'. Available: {available}")

    col_by_horizon = {"1M": "Fwd_1M", "3M": "Fwd_3M", "6M": "Fwd_6M"}
    rows = []
    for h_label, _, periods_per_yr in HORIZONS:
        col_name = col_by_horizon[h_label]
        valid = sub_df.dropna(subset=[col_name])
        rets = valid[col_name]

        count = len(rets)
        avg_ret = rets.mean() if count > 0 else np.nan
        ann_ret = ((1 + avg_ret) ** periods_per_yr - 1) if (pd.notna(avg_ret) and avg_ret > -1) else np.nan
        hit_rate = (rets > 0).mean() if count > 0 else np.nan
        ann_vol = (rets.std() * np.sqrt(periods_per_yr)) if count > 1 else np.nan
        max_dd = calculate_max_drawdown(rets)

        ic = np.nan
        if score_col and score_col in valid.columns:
            scores = valid[score_col].shift(1).dropna()
            aligned_rets = valid.loc[scores.index, col_name]
            if len(scores) > 2:
                ic, _ = spearmanr(scores, aligned_rets)

        avg_turnover = valid["Turnover"].mean() if not valid["Turnover"].isna().all() else np.nan

        rows.append({
            "horizon": h_label,
            "n": count,
            "avg_ret": avg_ret,
            "ann_ret": ann_ret,
            "hit_rate": hit_rate,
            "ann_vol": ann_vol,
            "max_dd": max_dd,
            "ic": ic,
            "avg_turnover": avg_turnover,
        })
    return rows


# ---------------------------------------------------------------------------
# Workbook writer
# ---------------------------------------------------------------------------
PCT_FMT = "0.00%"
NUM_FMT = "0.000"


def _write_stat_block(ws, row, section_label, regime_col, target_regime,
                       factor_col, df, expectation, conviction, is_first_block):
    """Writes one section header + 3 stat rows (1M/3M/6M) + blank separator.
    Returns the next free row."""
    ws.cell(row=row, column=1, value=section_label).font = Font(name="Arial", bold=True)

    if is_first_block:
        for j, header in enumerate(COLUMN_HEADERS, start=2):
            c = ws.cell(row=row, column=j, value=header)
            c.font = Font(name="Arial", bold=True)
            c.alignment = Alignment(horizontal="center", wrap_text=True)

    stats_rows = run_regime_factor_analysis(
        df, regime_col=regime_col, target_regime=target_regime, factor_col=factor_col
    )

    row += 1
    for stat in stats_rows:
        ws.cell(row=row, column=1, value=stat["horizon"]).font = Font(name="Arial", bold=True)
        ws.cell(row=row, column=2, value=expectation).font = Font(name="Arial")
        ws.cell(row=row, column=3, value=conviction).font = Font(name="Arial")
        ws.cell(row=row, column=4, value=None)  # Notes -- left for manual entry

        c = ws.cell(row=row, column=5, value=stat["n"])
        c.font = Font(name="Arial")

        for col_idx, key in [(6, "avg_ret"), (7, "ann_ret"), (8, "hit_rate"),
                              (9, "ann_vol"), (10, "max_dd")]:
            val = stat[key]
            c = ws.cell(row=row, column=col_idx,
                        value=None if pd.isna(val) else float(val))
            c.number_format = PCT_FMT
            c.font = Font(name="Arial")

        ic_val = stat["ic"]
        c = ws.cell(row=row, column=11, value=None if pd.isna(ic_val) else float(ic_val))
        c.number_format = NUM_FMT
        c.font = Font(name="Arial")

        to_val = stat["avg_turnover"]
        c = ws.cell(row=row, column=12, value=None if pd.isna(to_val) else float(to_val))
        c.number_format = PCT_FMT
        c.font = Font(name="Arial")

        row += 1
    row += 1  # blank separator row between blocks
    return row


def write_theme_sheet(ws, theme_name: str, df: pd.DataFrame, factor_col: str):
    ws["A1"] = theme_name
    ws["A1"].font = Font(name="Arial", size=18, bold=False)

    row = 3
    first_block = True

    # --- Section 1: Backtest_Octant breakdown (existing behavior) ---
    for regime_label in REGIME_ORDER:
        expectation, conviction = EXPECTATION_CONVICTION[theme_name][regime_label]
        row = _write_stat_block(
            ws, row, f"{theme_name} | {regime_label}", "Backtest_Octant",
            regime_label, factor_col, df, expectation, conviction, first_block,
        )
        first_block = False

    # --- Section 2: Base Regime breakdown (NEW) ---
    row += 1
    ws.cell(row=row, column=1,
            value=f"{theme_name} -- Base Regime Breakdown").font = Font(
        name="Arial", bold=True, italic=True, size=11)
    row += 2
    for regime_label in BASE_REGIMES:
        row = _write_stat_block(
            ws, row, f"{theme_name} | {regime_label} (Base Regime)",
            "Base_Regime_Confirmed", regime_label, factor_col, df,
            None, None, False,
        )

    # --- Section 3: Amplifier Direction breakdown (NEW) ---
    row += 1
    ws.cell(row=row, column=1,
            value=f"{theme_name} -- Amplifier Direction Breakdown").font = Font(
        name="Arial", bold=True, italic=True, size=11)
    row += 2
    for direction in AMPLIFIER_DIRECTIONS:
        row = _write_stat_block(
            ws, row, f"{theme_name} | {direction} (Amplifier)",
            "Amplifier_Direction", direction, factor_col, df,
            None, None, False,
        )

    # Footnote documenting assumptions (skill guidance: document assumptions
    # where the reader will see them).
    note_row = row + 1
    ws.cell(row=note_row, column=1,
            value=("Note: Information Coefficient and Average Turnover are blank -- "
                   "the input file has no factor z-score or portfolio weight column "
                   "to compute them. Expectation/Conviction are blank for the Base "
                   "Regime and Amplifier Direction blocks because the source PDF only "
                   "defines hypotheses at the Backtest_Octant level. Notes column is "
                   "left for manual entry.")
            ).font = Font(name="Arial", italic=True, size=9, color="808080")

    # Column widths
    widths = [30, 13, 12, 20, 10, 15, 14, 10, 15, 12, 16, 15]
    for idx, w in enumerate(widths, start=1):
        ws.column_dimensions[get_column_letter(idx)].width = w


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input", default="regime_factor_merged.csv")
    parser.add_argument("--output", default="QI-CAP_Theme_Regime_Stats.xlsx")
    args, _unknown = parser.parse_known_args()

    df = pd.read_csv(args.input)

    # --- NEW: Load existing workbook to preserve manual columns ---
    if os.path.exists(args.output):
        print(f"Loading existing {args.output} to preserve manual formulas...")
        wb = load_workbook(args.output)
    else:
        wb = Workbook()
        wb.remove(wb.active)

    def get_sheet(name):
        return wb[name] if name in wb.sheetnames else wb.create_sheet(title=name)
    # --------------------------------------------------------------

    # Absolute-return sheets
    for theme_name, factor_col in THEME_TO_FACTOR_COL.items():
        ws = get_sheet(theme_name)
        write_theme_sheet(ws, theme_name, df, factor_col=factor_col)

    # Relative-return sheets
    for theme_name, factor_col in THEME_TO_ACROSS_RELATIVE_FACTOR_COL.items():
        ws = get_sheet(f"{theme_name} (Across Relative)")
        write_theme_sheet(ws, theme_name, df, factor_col=factor_col)

    for theme_name, factor_col in THEME_TO_SELF_RELATIVE_COL.items():
        ws = get_sheet(f"{theme_name} (Self Relative)")
        write_theme_sheet(ws, theme_name, df, factor_col=factor_col)

    wb.save(args.output)
    print("Update complete. Manual formulas preserved.")

if __name__ == "__main__":
    main()


if __name__ == "__main__":
    main()

Loading existing QI-CAP_Theme_Regime_Stats.xlsx to preserve manual formulas...
Update complete. Manual formulas preserved.
Loading existing QI-CAP_Theme_Regime_Stats.xlsx to preserve manual formulas...
Update complete. Manual formulas preserved.


In [283]:
#dashboard_and_distributions

"""
Adds two things to the existing QI-CAP_Theme_Regime_Stats.xlsx workbook:

  1. A new FIRST sheet ("Summary Dashboard") -- heatmaps of Avg 1M Return,
     Hit Rate, and N (months) across all 16 theme sheets x the 6
     Backtest_Octant buckets.

  2. On each of the 16 existing sheets, next to every regime block's "1M"
     row (Octant, Base Regime, AND Amplifier Direction blocks alike):
       - Mean, Median, Std Dev, Normality (Shapiro-Wilk), Shapiro p-value,
         and N of the underlying monthly returns
       - a small labeled smooth density curve (KDE, not a bar histogram) of
         that same return distribution, titled with the regime name

Only the 1M horizon is touched -- 3M/6M rows are untouched.

This modifies the workbook produced by build_regime_factor_workbook_v2.py
IN PLACE (loads it, adds content, re-saves to the same path) -- it does not
rebuild the existing stat blocks.

Usage (run after build_regime_factor_workbook_v2.py has produced the
workbook):
    python add_dashboard_and_distributions.py \
        --data regime_factor_merged.csv \
        --workbook QI-CAP_Theme_Regime_Stats.xlsx
"""

import argparse
import numpy as np
import pandas as pd
from scipy.stats import shapiro, gaussian_kde
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment
from openpyxl.formatting.rule import ColorScaleRule
from openpyxl.chart import ScatterChart, Series, Reference
from openpyxl.chart.marker import Marker
from openpyxl.chart.data_source import NumData, NumVal
from openpyxl.utils import get_column_letter

DATE_COL = "Month_End"

REGIME_ORDER = [
    "Goldilocks Tailwind",
    "Goldilocks Headwind",
    "Reflation Headwind",
    "Reflation Tailwind",
    "Stagflation Tailwind",
    "Deflation Tailwind",
]
BASE_REGIMES = ["Goldilocks", "Reflation", "Stagflation", "Deflation"]
AMPLIFIER_DIRECTIONS = ["Tailwind", "Headwind"]

THEME_TO_FACTOR_COL = {
    "Value": "value", "Momentum": "momentum", "Profitability": "profitability",
    "Quality": "quality", "Earnings": "earnings", "Sentiment": "sentiment main",
    "Reversal": "Reversal", "Management": "management",
}
# FIX: was 'THEME_TO_RELATIVE_FACTOR_COL' with values like "value_relative" --
# that column never existed (cell 1 generates "value_across_relative", with
# _across_), and two functions below (process_theme_sheet, build_dashboard)
# already referenced the correctly-named THEME_TO_ACROSS_RELATIVE_FACTOR_COL,
# which was never defined anywhere -- an immediate NameError on both paths.
# Renamed + values corrected to match cell 1's actual output columns and
# cell 3's naming.
THEME_TO_ACROSS_RELATIVE_FACTOR_COL = {
    "Value": "value_across_relative", "Momentum": "momentum_across_relative",
    "Profitability": "profitability_across_relative", "Quality": "quality_across_relative",
    "Earnings": "earnings_across_relative",
    "Management": "management_across_relative",
}

THEME_TO_SELF_RELATIVE_COL = {"Value": "value_theme_relative",
    "Momentum": "momentum_theme_relative",
    "Profitability": "profitability_theme_relative",
    "Quality": "quality_theme_relative",
    "Earnings": "earnings_theme_relative",
    "Sentiment": "sentiment main_theme_relative",
    "Reversal": "Reversal_theme_relative",
    "Management": "management_theme_relative",
}

# Layout constants -- new stat columns start right after the existing 12
# (A..L), leaving column M as a spacer. Chart sits after those. Histogram
# bin data is written far to the right and hidden -- it has to live in real
# cells for the chart to reference, but nobody needs to see it.
DIST_COL_START = 14        # column N
DIST_HEADERS = ["Mean (1M)", "Median (1M)", "Std Dev (1M)",
                 "Normality (1M)", "Shapiro p (1M)", "N (dist check)"]
CHART_ANCHOR_COL = 21      # column U
HIST_HELPER_COL_START = 40  # hidden helper area holding the KDE curve's (x,y) grid
KDE_GRID_POINTS = 60


# ---------------------------------------------------------------------------
# Data helpers
# ---------------------------------------------------------------------------
def load_merged(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df[DATE_COL] = pd.to_datetime(df[DATE_COL])
    return df.sort_values(DATE_COL).reset_index(drop=True)


def get_1m_returns(df: pd.DataFrame, regime_col: str, target_regime: str, factor_col: str) -> pd.Series:
    """Same 1-month lag as the rest of the pipeline: regime at month t governs
    the return realized at month t+1."""
    lagged = df[regime_col].shift(1)
    return df.loc[lagged == target_regime, factor_col].dropna()


def distribution_stats(returns: pd.Series) -> dict:
    n = len(returns)
    mean = returns.mean() if n else np.nan
    median = returns.median() if n else np.nan
    std = returns.std() if n > 1 else np.nan

    p_value = np.nan
    if n >= 3 and returns.std(ddof=0) > 0:
        try:
            _, p_value = shapiro(returns)
        except Exception:
            p_value = np.nan

    if n < 3:
        normal_label = "N/A (n<3)"
    elif np.isnan(p_value):
        normal_label = "N/A"
    else:
        normal_label = "Normal" if p_value > 0.05 else "Not Normal"

    return {"n": n, "mean": mean, "median": median, "std": std,
            "shapiro_p": p_value, "normal_label": normal_label}


def parse_block_label(label):
    """'Theme | Octant' -> ('Theme','Backtest_Octant','Octant')
    'Theme | X (Base Regime)' -> ('Theme','Base_Regime_Confirmed','X')
    'Theme | X (Amplifier)'   -> ('Theme','Amplifier_Direction','X')
    Returns None if this isn't a regime-block label row."""
    if not label or " | " not in label:
        return None
    theme_part, regime_part = label.split(" | ", 1)
    regime_part = regime_part.strip()
    if regime_part.endswith("(Base Regime)"):
        return theme_part.strip(), "Base_Regime_Confirmed", regime_part.replace("(Base Regime)", "").strip()
    if regime_part.endswith("(Amplifier)"):
        return theme_part.strip(), "Amplifier_Direction", regime_part.replace("(Amplifier)", "").strip()
    return theme_part.strip(), "Backtest_Octant", regime_part


# ---------------------------------------------------------------------------
# Per-sheet: distribution stats + histogram next to each regime's 1M row
# ---------------------------------------------------------------------------
def add_distribution_block(ws, df, row_1m, label_row, block_idx, factor_col):
    parsed = parse_block_label(ws.cell(row=label_row, column=1).value)
    if parsed is None:
        return
    _, regime_col, target_regime = parsed

    returns = get_1m_returns(df, regime_col, target_regime, factor_col)
    stats = distribution_stats(returns)

    if block_idx == 0:  # header once per sheet, same row as the original column headers
        for j, header in enumerate(DIST_HEADERS, start=DIST_COL_START):
            c = ws.cell(row=label_row, column=j, value=header)
            c.font = Font(name="Arial", bold=True)
            c.alignment = Alignment(horizontal="center", wrap_text=True)

    values = [stats["mean"], stats["median"], stats["std"],
              stats["normal_label"], stats["shapiro_p"], stats["n"]]
    for offset, val in enumerate(values):
        j = DIST_COL_START + offset
        write_val = None if (isinstance(val, float) and pd.isna(val)) else val
        c = ws.cell(row=row_1m, column=j, value=write_val)
        c.font = Font(name="Arial")
        if offset in (0, 1, 2):
            c.number_format = "0.00%"
        elif offset == 4:
            c.number_format = "0.000"

    # Small smooth density curve (KDE, not a histogram), anchored on the
    # same row, right of the stat columns
    if stats["n"] >= 2 and returns.std(ddof=0) > 0:
        try:
            kde = gaussian_kde(returns.to_numpy())
            pad = 2 * returns.std()
            grid_x = np.linspace(returns.min() - pad, returns.max() + pad, KDE_GRID_POINTS)
            grid_y = kde(grid_x)
        except Exception:
            grid_x = None

        if grid_x is not None:
            x_col = HIST_HELPER_COL_START + block_idx * 3
            y_col = x_col + 1
            for i in range(len(grid_x)):
                ws.cell(row=3 + i, column=x_col, value=float(grid_x[i]))
                ws.cell(row=3 + i, column=y_col, value=float(grid_y[i]))

            x_ref = Reference(ws, min_col=x_col, min_row=3, max_row=3 + len(grid_x) - 1)
            y_ref = Reference(ws, min_col=y_col, min_row=3, max_row=3 + len(grid_x) - 1)

            chart = ScatterChart()
            chart.legend = None
            chart.title = target_regime
            chart.x_axis.title = "Return"
            chart.y_axis.title = "Density"
            chart.x_axis.numFmt = "0.0%"
            chart.x_axis.delete = False
            chart.y_axis.delete = False
            chart.x_axis.majorGridlines = None
            chart.y_axis.majorGridlines = None
            chart.width = 5.4
            chart.height = 2.9
            # See note above about hidden helper columns and plotVisOnly.
            chart.visible_cells_only = False

            series = Series(y_ref, x_ref, title=None)
            series.marker = Marker(symbol="none")
            series.smooth = True
            series.graphicalProperties.line.width = 20000
            series.graphicalProperties.line.solidFill = "1F4E79"

            # Cache values directly (see note above) so this renders in any
            # viewer, not just Excel's own recalculation on open.
            series.xVal.numRef.numCache = NumData(
                pt=[NumVal(idx=i, v=float(grid_x[i])) for i in range(len(grid_x))],
                ptCount=len(grid_x),
            )
            series.yVal.numRef.numCache = NumData(
                pt=[NumVal(idx=i, v=float(grid_y[i])) for i in range(len(grid_y))],
                ptCount=len(grid_y),
            )
            chart.series.append(series)

            ws.add_chart(chart, f"{get_column_letter(CHART_ANCHOR_COL)}{row_1m}")


def process_theme_sheet(ws, df):
    # Clear old charts so they don't visually stack upon updating
    ws._charts = []  
    is_across_relative = ws.title.strip().endswith("(Across Relative)")
    is_self_relative = ws.title.strip().endswith("(Self Relative)")
    across_base_theme = ws.title.replace(" (Across Relative)", "").strip()
    self_base_theme = ws.title.replace(" (Self Relative)", "").strip()

    if is_across_relative:
        factor_col = (THEME_TO_ACROSS_RELATIVE_FACTOR_COL)[across_base_theme]
    elif is_self_relative:
        factor_col = (THEME_TO_SELF_RELATIVE_COL)[self_base_theme]
    else: 
        factor_col = (THEME_TO_FACTOR_COL)[across_base_theme]

    block_idx = 0
    for r in range(1, ws.max_row + 1):
        if ws.cell(row=r, column=1).value == "1M":
            add_distribution_block(ws, df, row_1m=r, label_row=r - 1,
                                    block_idx=block_idx, factor_col=factor_col)
            block_idx += 1

    for j in range(DIST_COL_START, DIST_COL_START + len(DIST_HEADERS)):
        ws.column_dimensions[get_column_letter(j)].width = 14

    last_helper_col = HIST_HELPER_COL_START + block_idx * 3
    for j in range(HIST_HELPER_COL_START, last_helper_col + 1):
        ws.column_dimensions[get_column_letter(j)].hidden = True


# ---------------------------------------------------------------------------
# Summary Dashboard sheet -- all three regime groupings (Octant, Base Regime,
# Amplifier Direction) side by side, grouped under merged headers
# ---------------------------------------------------------------------------
REGIME_GROUPS = [
    ("Backtest Octant", "Backtest_Octant", REGIME_ORDER),
    ("Base Regime", "Base_Regime_Confirmed", BASE_REGIMES),
    ("Amplifier Direction", "Amplifier_Direction", AMPLIFIER_DIRECTIONS),
]


def build_dashboard(wb, df):
    ws = wb.create_sheet(title="Summary Dashboard", index=0)
    ws["A1"] = "1M Regime Summary Dashboard"
    ws["A1"].font = Font(name="Arial", size=18)
    ws["A2"] = ("Average 1-month forward return, hit rate, and sample size by theme, "
                "across Backtest_Octant, Base Regime, and Amplifier Direction "
                "(1-month lag applied throughout).")
    ws["A2"].font = Font(name="Arial", italic=True, size=10, color="808080")

    theme_labels = list(THEME_TO_FACTOR_COL.keys()) + [f"{t} (Across Relative)" for t in THEME_TO_ACROSS_RELATIVE_FACTOR_COL] + [f"{t} (Self Relative)" for t in THEME_TO_SELF_RELATIVE_COL]
    factor_lookup = dict(THEME_TO_FACTOR_COL)
    factor_lookup.update({f"{t} (Across Relative)": c for t, c in THEME_TO_ACROSS_RELATIVE_FACTOR_COL.items()})
    factor_lookup.update({f"{t} (Self Relative)": c for t, c in THEME_TO_SELF_RELATIVE_COL.items()})

    def write_table(start_row, title, value_fn, number_format, color_rule_factory):
        ws.cell(row=start_row, column=1, value=title).font = Font(name="Arial", bold=True)
        group_header_row = start_row + 1
        col_header_row = start_row + 2
        data_start_row = start_row + 3

        col = 2
        group_ranges = []  # (start_col, end_col, regime_col, targets)
        for group_label, regime_col, targets in REGIME_GROUPS:
            start_col = col
            end_col = col + len(targets) - 1

            ws.merge_cells(start_row=group_header_row, start_column=start_col,
                            end_row=group_header_row, end_column=end_col)
            gc = ws.cell(row=group_header_row, column=start_col, value=group_label)
            gc.font = Font(name="Arial", bold=True, italic=True, size=10)
            gc.alignment = Alignment(horizontal="center")

            for j, target in enumerate(targets, start=start_col):
                c = ws.cell(row=col_header_row, column=j, value=target)
                c.font = Font(name="Arial", bold=True)
                c.alignment = Alignment(horizontal="center", wrap_text=True)

            group_ranges.append((start_col, end_col, regime_col, targets))
            col = end_col + 2  # one spacer column between groups

        for i, theme_label in enumerate(theme_labels, start=data_start_row):
            ws.cell(row=i, column=1, value=theme_label).font = Font(name="Arial")
            for start_col, end_col, regime_col, targets in group_ranges:
                for j, target in zip(range(start_col, end_col + 1), targets):
                    returns = get_1m_returns(df, regime_col, target, factor_lookup[theme_label])
                    val = value_fn(returns)
                    c = ws.cell(row=i, column=j, value=val)
                    c.number_format = number_format
                    c.font = Font(name="Arial")

        end_row = data_start_row + len(theme_labels) - 1
        if color_rule_factory is not None:
            for start_col, end_col, _, _ in group_ranges:
                # Fresh rule instance per range -- ties each group's color
                # scale to its own min/max rather than one shared scale
                # across octant/base/amplifier, which have different spreads.
                rule = color_rule_factory()
                rng = f"{get_column_letter(start_col)}{data_start_row}:{get_column_letter(end_col)}{end_row}"
                ws.conditional_formatting.add(rng, rule)

        last_col = group_ranges[-1][1]
        return end_row, last_col

    row = 4
    row, last_col = write_table(
        row, "Avg 1M Return", lambda r: (r.mean() if len(r) else None), "0.00%",
        lambda: ColorScaleRule(start_type="min", start_color="F8696B",
                                mid_type="num", mid_value=0, mid_color="FFFFFF",
                                end_type="max", end_color="63BE7B"),
    )
    row += 4
    row, last_col = write_table(
        row, "Hit Rate (1M)", lambda r: ((r > 0).mean() if len(r) else None), "0.0%",
        lambda: ColorScaleRule(start_type="num", start_value=0, start_color="F8696B",
                                mid_type="num", mid_value=0.5, mid_color="FFFFFF",
                                end_type="num", end_value=1, end_color="63BE7B"),
    )
    row += 4
    row, last_col = write_table(row, "N (months)", lambda r: len(r), "0", None)

    ws.column_dimensions["A"].width = 26
    for j in range(2, last_col + 1):
        ws.column_dimensions[get_column_letter(j)].width = 15
    ws.freeze_panes = "B7"

    ws.cell(row=row + 2, column=1, value=(
        "Note: each table repeats the same theme x metric view across three regime "
        "groupings -- Backtest Octant (finest), Base Regime, and Amplifier Direction "
        "(coarsest) -- color-scaled independently per grouping. Several buckets rest "
        "on very few independent regime spells (see the spell-count discussion) -- "
        "check the N table above, and each sheet's own N, before reading small "
        "cross-cell differences as meaningful."
    )).font = Font(name="Arial", italic=True, size=9, color="808080")


# ---------------------------------------------------------------------------
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", default="regime_factor_merged.csv")
    parser.add_argument("--workbook", default="QI-CAP_Theme_Regime_Stats.xlsx")
    args, _unknown = parser.parse_known_args()  # tolerant of Jupyter's injected -f flag

    df = load_merged(args.data)
    wb = load_workbook(args.workbook)

    build_dashboard(wb, df)

    for sheet_name in [s for s in wb.sheetnames if not s.startswith("Summary Dashboard")]:
        process_theme_sheet(wb[sheet_name], df)
# --- NEW: Delete rogue Summary Dashboards (e.g., "Summary Dashboard1") ---
    for sheet_name in wb.sheetnames:
        if sheet_name.startswith("Summary Dashboard") and sheet_name != "Summary Dashboard":
            del wb[sheet_name]
            print(f"Cleaned up duplicate sheet: {sheet_name}")
    # -------------------------------------------------------------------------
    wb.save(args.workbook)
    print(f"Updated {args.workbook}: added 'Summary Dashboard' + 1M distribution "
          f"stats/charts across {len(wb.sheetnames) - 1} theme sheets.")


if __name__ == "__main__":
    main()

Cleaned up duplicate sheet: Summary Dashboard1
Updated QI-CAP_Theme_Regime_Stats.xlsx: added 'Summary Dashboard' + 1M distribution stats/charts across 22 theme sheets.


In [284]:
import pandas as pd
import numpy as np

def compound(series):
    """Geometrically compound a series of returns"""
    return np.prod(1 + series) - 1

# 1. Load the merged data
df = pd.read_csv("regime_factor_merged.csv")
df["Month_End"] = pd.to_datetime(df["Month_End"])
df = df.sort_values("Month_End").reset_index(drop=True)

# 2. Apply the 1-month lag to the classifier
df["Lagged_Octant"] = df["Backtest_Octant"].shift(1)
df = df.dropna(subset=["Lagged_Octant"]).copy()

# 3. The Spell ID Trick: Create a unique ID for every continuous block of the same regime
# Every time the lagged octant changes from the previous row, the cumulative sum goes up by 1
df["Spell_ID"] = (df["Lagged_Octant"] != df["Lagged_Octant"].shift(1)).cumsum()

# Define the valid 6 octants and the Self-Relative columns
VALID_OCTANTS = [
    "Goldilocks Tailwind", "Goldilocks Headwind",
    "Reflation Headwind", "Reflation Tailwind",
    "Stagflation Tailwind", "Deflation Tailwind"
]
themes = [
    "value", "momentum", "profitability", "quality", 
    "earnings", "sentiment main", "Reversal", "management"
]

print("=========================================================================")
print("                  SPELL-LEVEL PERSISTENCE TEST (SELF RELATIVE)           ")
print("=========================================================================\n")

# Loop through each theme and evaluate its Self-Relative spell persistence
for theme in themes:
    col_name = f"{theme}_theme_relative"
    if col_name not in df.columns:
        continue
        
    # Group the dataframe by the unique Spell ID and the Regime name
    spell_summary = df.groupby(["Spell_ID", "Lagged_Octant"]).agg(
        Months_in_Spell=(col_name, "count"),
        Spell_Cumulative_Return=(col_name, compound),
        Positive_Months=(col_name, lambda x: (x > 0).sum())
    ).reset_index()

    # Determine if the spell was a "Hit" (total compounded relative return > 0)
    spell_summary["Spell_Hit"] = spell_summary["Spell_Cumulative_Return"] > 0
    
    # Filter only for the 6 valid octants
    spell_summary = spell_summary[spell_summary["Lagged_Octant"].isin(VALID_OCTANTS)]

    # Roll up the spell data into a clean regime-level summary
    regime_stats = spell_summary.groupby("Lagged_Octant").agg(
        Total_Spells=("Spell_ID", "count"),
        Total_Months=("Months_in_Spell", "sum"),
        Spell_Hit_Rate=("Spell_Hit", "mean"),
        Monthly_Hit_Rate=("Positive_Months", lambda x: x.sum() / spell_summary.loc[x.index, "Months_in_Spell"].sum())
    ).reindex(VALID_OCTANTS) # Keep the order consistent

    # Format the table for printing
    formatted_stats = pd.DataFrame({
        "Total Spells": regime_stats["Total_Spells"],
        "Total Months": regime_stats["Total_Months"],
        "Avg Length": (regime_stats["Total_Months"] / regime_stats["Total_Spells"]).round(1).astype(str) + " mo",
        "Monthly Hit Rate": (regime_stats["Monthly_Hit_Rate"] * 100).round(1).astype(str) + "%",
        "SPELL HIT RATE": (regime_stats["Spell_Hit_Rate"] * 100).round(1).astype(str) + "%"
    })
    
    print(f"--- THEME: {theme.upper()} (Self Relative) ---")
    print(formatted_stats.to_markdown())
    print("\n")

                  SPELL-LEVEL PERSISTENCE TEST (SELF RELATIVE)           

--- THEME: VALUE (Self Relative) ---
| Lagged_Octant        |   Total Spells |   Total Months | Avg Length   | Monthly Hit Rate   | SPELL HIT RATE   |
|:---------------------|---------------:|---------------:|:-------------|:-------------------|:-----------------|
| Goldilocks Tailwind  |             10 |             17 | 1.7 mo       | 64.7%              | 70.0%            |
| Goldilocks Headwind  |              2 |             19 | 9.5 mo       | 57.9%              | 50.0%            |
| Reflation Headwind   |              3 |             20 | 6.7 mo       | 40.0%              | 33.3%            |
| Reflation Tailwind   |              5 |             21 | 4.2 mo       | 47.6%              | 60.0%            |
| Stagflation Tailwind |              5 |             16 | 3.2 mo       | 37.5%              | 20.0%            |
| Deflation Tailwind   |             10 |             38 | 3.8 mo       | 52.6%           

In [285]:
import pandas as pd
import numpy as np

# 1. Load the merged data
df = pd.read_csv("regime_factor_merged.csv")
df["Month_End"] = pd.to_datetime(df["Month_End"])
df = df.sort_values("Month_End").reset_index(drop=True)

# 2. Apply the 1-month lag to the classifier
df["Lagged_Octant"] = df["Backtest_Octant"].shift(1)
df = df.dropna(subset=["Lagged_Octant"]).copy()

# 3. Create Spell IDs to ensure correlation does not bleed across regime changes
df["Spell_ID"] = (df["Lagged_Octant"] != df["Lagged_Octant"].shift(1)).cumsum()

VALID_OCTANTS = [
    "Goldilocks Tailwind", "Goldilocks Headwind",
    "Reflation Headwind", "Reflation Tailwind",
    "Stagflation Tailwind", "Deflation Tailwind"
]

themes = [
    "value", "momentum", "profitability", "quality", 
    "earnings", "sentiment main", "Reversal", "management"
]

print("=========================================================================")
print("           INTRA-REGIME AUTOCORRELATION (1-MONTH TREND)                  ")
print("=========================================================================\n")

for theme in themes:
    col_name = f"{theme}_theme_relative"
    if col_name not in df.columns:
        continue
        
    # Safely shift the return by 1 month STRICTLY within the same spell block
    df[f"{col_name}_lag1"] = df.groupby("Spell_ID")[col_name].shift(1)
    
    results = []
    for octant in VALID_OCTANTS:
        # Filter for the specific regime
        subset = df[df["Lagged_Octant"] == octant]
        
        # Calculate Pearson correlation between current month and previous month
        corr = subset[col_name].corr(subset[f"{col_name}_lag1"])
        
        # Count how many valid month-to-month pairs we actually have
        n_pairs = subset[[col_name, f"{col_name}_lag1"]].dropna().shape[0]
        
        results.append({
            "Regime": octant,
            "Pearson Correlation": corr,
            "Valid Pairs (N)": n_pairs
        })
        
    res_df = pd.DataFrame(results)
    
    # Clean up the formatting for readability
    res_df["Pearson Correlation"] = res_df["Pearson Correlation"].apply(
        lambda x: f"{x:+.3f}" if pd.notna(x) else "N/A"
    )
    
    print(f"--- THEME: {theme.upper()} (Self Relative) ---")
    print(res_df.to_markdown(index=False))
    print("\n")

           INTRA-REGIME AUTOCORRELATION (1-MONTH TREND)                  

--- THEME: VALUE (Self Relative) ---
| Regime               |   Pearson Correlation |   Valid Pairs (N) |
|:---------------------|----------------------:|------------------:|
| Goldilocks Tailwind  |                -0.825 |                 7 |
| Goldilocks Headwind  |                -0.18  |                17 |
| Reflation Headwind   |                 0.569 |                17 |
| Reflation Tailwind   |                 0.124 |                16 |
| Stagflation Tailwind |                -0.222 |                11 |
| Deflation Tailwind   |                 0.367 |                28 |


--- THEME: MOMENTUM (Self Relative) ---
| Regime               |   Pearson Correlation |   Valid Pairs (N) |
|:---------------------|----------------------:|------------------:|
| Goldilocks Tailwind  |                -0.443 |                 7 |
| Goldilocks Headwind  |                -0.629 |                17 |
| Reflation Headwi